# Initial convergence studies

This notebook reads completed numerical records and their inspected figures. It performs no PDE solves. Each series declares whether it changes the macro mesh, local mesh, trace partition or time step. Exact analytical errors and numerical refinement increments have separate labels.

The catalogue retains incomplete levels and unresolved references. Three coarse observations do not establish asymptotic rates or uniform stability. Full historical reproductions and broad parameter sweeps are separate from these initial studies.
The source-only catalogue validates all retained scalar JSON identities and renders selected curves directly from those records. It does not recompute their physical acquisitions. `--historical` additionally requires the original inspected figure payloads; individual physical notebooks and declared complete recipes perform fresh numerical acquisitions.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/12898e25eb1d08304e055bd84fb08524e056b17594a6f676e21c1294f3dcc1f7/73_initial_convergence-companion.zip"
COMPANION_SHA256 = "12898e25eb1d08304e055bd84fb08524e056b17594a6f676e21c1294f3dcc1f7"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("convergence/73_initial_convergence.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"


In [ ]:
from pathlib import Path
import hashlib
import json
from IPython.display import Image, Markdown, display

catalogue = json.loads((root / "examples/results/minimal-convergence/catalogue.json").read_text())
entries = catalogue["entries"]
for entry in entries:
    identities = [("public_record", "record_sha256")]
    if HISTORICAL_REPLAY:
        identities.append(("figure", "figure_sha256"))
    for key, identity in identities:
        with (root / entry[key]).open("rb") as stream:
            assert hashlib.file_digest(stream, "sha256").hexdigest() == entry[identity]
lines = ["| Study | Refinement | Observed conclusion |", "| --- | --- | --- |"]
for entry in entries:
    lines.append(f"| {entry['title']} | {entry['refinement']} | {entry['summary']} |")
display(Markdown("\n".join(lines)))
print(f"Validated {len(entries)} completed scalar study records. Original figure hashes are checked only for --historical replay.")

The panels below show an analytical polynomial-family control, a heterogeneous Darcy case whose flux has not converged, the analytical elastic wave, and a temporal control on a fixed three-layer spatial mesh. Their numerical records contain the separate field norms and original-equation diagnostics.

In [ ]:
from examples.initial_study_gallery import plot_panels
from examples.minimal_wave_convergence import plot as plot_wave_record

selected = ("scalar-mh2m", "darcy-spe10", "wave-elastic-wave", "wave-three-layer")
by_id = {entry["id"]: entry for entry in entries}
figure_directory = root / "build/notebook-figures/initial-convergence"
figure_directory.mkdir(parents=True, exist_ok=True)
for name in selected:
    entry = by_id[name]
    display(Markdown(f"### {entry['title']}\n\n{entry['summary']}\n\n{entry['limitations']}"))
    if HISTORICAL_REPLAY:
        figure = root / entry["figure"]
    else:
        observation = json.loads((root / entry["public_record"]).read_text())
        figure = figure_directory / f"{name}.png"
        if "panels" in observation:
            plot_panels(observation, figure)
        elif name == "darcy-spe10":
            # These are stored physical norms; no exact pressure is available.
            rows = [{"level": row["level"], "norms": row["norms"][str(max(row["norm_orders"]))]}
                    for row in observation["rows"]]
            display_record = {"rows": rows,
                "plot_fields": ["pressure_l2_norm", "flux_l2_norm"],
                "level_label": "Declared local refinement",
                "norm_label": "Physical L2 norm (not exact-solution error)",
                "figure_title": "Retained SPE10 observations"}
            plot_wave_record(display_record, figure)
        else:
            plot_wave_record(observation, figure)
    display(Image(filename=str(figure), width=893))
print("Figures above render retained scalar observations; PDE solves during rendering: 0.")
